# World Cup 2026 Match Predictor

A from-scratch Poisson attack/defense model with a Dixon-Coles correction, fitted on international football results (2023-2026) and applied to live 2026 World Cup Round of 16 fixtures.

## 1. Load the data

In [2]:
import pandas as pd

# Load the three raw datasets
# Update these paths to wherever you've saved the CSVs locally
results = pd.read_csv("results.csv")
shootouts = pd.read_csv("shootouts.csv")
goalscorers = pd.read_csv("goalscorers.csv")

# Parse date columns properly (they'll load as strings otherwise)
results["date"] = pd.to_datetime(results["date"])
shootouts["date"] = pd.to_datetime(shootouts["date"])
goalscorers["date"] = pd.to_datetime(goalscorers["date"])

# Quick sanity checks before we do anything else
print("results shape:", results.shape)
print("date range:", results["date"].min(), "to", results["date"].max())
print(results.head())

results shape: (49502, 9)
date range: 1872-11-30 00:00:00 to 2026-07-09 00:00:00
        date home_team away_team  home_score  away_score tournament     city  \
0 1872-11-30  Scotland   England         0.0         0.0   Friendly  Glasgow   
1 1873-03-08   England  Scotland         4.0         2.0   Friendly   London   
2 1874-03-07  Scotland   England         2.0         1.0   Friendly  Glasgow   
3 1875-03-06   England  Scotland         2.0         2.0   Friendly   London   
4 1876-03-04  Scotland   England         3.0         0.0   Friendly  Glasgow   

    country  neutral  
0  Scotland    False  
1   England    False  
2  Scotland    False  
3   England    False  
4  Scotland    False  


## 2. Filter to a recent, relevant time window (2023 onward)

In [3]:
# Filter to matches from 2023 onwards
cutoff_date = "2023-01-01"
results_recent = results[results["date"] >= cutoff_date].copy()

print("Filtered shape:", results_recent.shape)
print("Filtered date range:", results_recent["date"].min(), "to", results_recent["date"].max())

Filtered shape: (3695, 9)
Filtered date range: 2023-01-02 00:00:00 to 2026-07-09 00:00:00


## 3. Check for scheduled-but-unplayed fixtures (these become our live prediction targets)

In [4]:
# Look at the most recent rows to check for scheduled/future matches
print(results_recent.tail(10))

# Check for missing scores (a sign of fixture placeholders, not real results)
print("Missing home_score:", results_recent["home_score"].isna().sum())
print("Missing away_score:", results_recent["away_score"].isna().sum())

            date      home_team away_team  home_score  away_score  \
49492 2026-07-03       Colombia     Ghana         1.0         0.0   
49493 2026-07-04         Canada   Morocco         0.0         3.0   
49494 2026-07-04       Paraguay    France         0.0         1.0   
49495 2026-07-05         Brazil    Norway         NaN         NaN   
49496 2026-07-05         Mexico   England         NaN         NaN   
49497 2026-07-06       Portugal     Spain         NaN         NaN   
49498 2026-07-06  United States   Belgium         NaN         NaN   
49499 2026-07-06      Argentina     Egypt         NaN         NaN   
49500 2026-07-06    Switzerland  Colombia         NaN         NaN   
49501 2026-07-09         France   Morocco         NaN         NaN   

           tournament             city        country  neutral  
49492  FIFA World Cup      Kansas City  United States     True  
49493  FIFA World Cup          Houston  United States     True  
49494  FIFA World Cup     Philadelphia  Unite

## 4. Split into completed matches (training data) vs upcoming fixtures (prediction targets)

In [5]:
 # Split into: completed matches (for training) vs upcoming fixtures (for prediction)
# Why this split matters:
# - `completed` = matches with real final scores. This is our ground truth,
#   used to estimate team strength (Elo/Poisson parameters) and to backtest
#   how good our model is (predict past matches, compare to what actually happened).
# - `upcoming_fixtures` = matches already scheduled but not yet played (NaN scores).
#   These are NOT used for training — they're the actual output of the whole
#   project: real predictions for real future matches we can check against
#   reality once they're played.

completed = results_recent[results_recent["home_score"].notna()].copy()
upcoming_fixtures = results_recent[results_recent["home_score"].isna()].copy()

# Sanity check: row counts should add up to the full filtered dataset
assert len(completed) + len(upcoming_fixtures) == len(results_recent), \
    "Row counts don't add up — something's wrong with the split"

print("Completed matches (training data):", completed.shape)
print("Upcoming fixtures (prediction targets):", upcoming_fixtures.shape)
print()
print("Upcoming fixtures detail:")
print(upcoming_fixtures[["date", "home_team", "away_team", "city", "neutral"]].to_string(index=False))

Completed matches (training data): (3688, 9)
Upcoming fixtures (prediction targets): (7, 9)

Upcoming fixtures detail:
      date     home_team away_team            city  neutral
2026-07-05        Brazil    Norway East Rutherford     True
2026-07-05        Mexico   England     Mexico City    False
2026-07-06      Portugal     Spain          Dallas     True
2026-07-06 United States   Belgium         Seattle    False
2026-07-06     Argentina     Egypt         Atlanta     True
2026-07-06   Switzerland  Colombia       Vancouver     True
2026-07-09        France   Morocco      Foxborough     True


## 5. Feature engineering: rolling form

For each team, compute recent scoring/conceding form over their last 5 matches, using only prior matches (no leakage).

In [6]:
# Rolling form: for each team, before each match, calculate their recent scoring
# and conceding average over their last N matches (not including the match itself).
#
# This matters because a team's form right now often predicts near-term performance
# better than their overall long-run average. Two teams could have the same
# average goals scored across the whole 2023-2026 window, but if one is in a
# slump the last 5 games and the other is on a hot streak, that's a real signal
# our model should pick up.
#
# The tricky part: our dataset is "wide" per match (one row = home_team + away_team),
# but we need "long" per team-appearance to compute a rolling average per team.
# So the approach is:
#   1. Reshape into one row per team per match (a team-centric view)
#   2. Sort by team and date
#   3. Compute a rolling average using only PAST matches (shift by 1 to exclude
#      the current match itself, otherwise we'd be leaking the outcome into
#      its own feature)
#   4. Reshape back and merge onto the original home/away structure

def build_team_match_view(df):
    # Home team's perspective: goals_for = home_score, goals_against = away_score
    home_view = df[["date", "home_team", "home_score", "away_score", "tournament"]].copy()
    home_view.columns = ["date", "team", "goals_for", "goals_against", "tournament"]

    # Away team's perspective: goals_for = away_score, goals_against = home_score
    away_view = df[["date", "away_team", "away_score", "home_score", "tournament"]].copy()
    away_view.columns = ["date", "team", "goals_for", "goals_against", "tournament"]

    # Stack both perspectives into one long dataframe: one row per team per match
    team_view = pd.concat([home_view, away_view], ignore_index=True)
    team_view = team_view.sort_values(["team", "date"]).reset_index(drop=True)
    return team_view

team_view = build_team_match_view(completed)

# Rolling average of last 5 matches, per team, using only prior matches.
# .shift(1) is the key line: without it, the "rolling average" for a match
# would include that match's own result, which defeats the purpose (we want
# to know form GOING INTO the match, not form including it).
WINDOW = 5

team_view["rolling_goals_for"] = (
    team_view.groupby("team")["goals_for"]
    .apply(lambda x: x.shift(1).rolling(WINDOW, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)

team_view["rolling_goals_against"] = (
    team_view.groupby("team")["goals_against"]
    .apply(lambda x: x.shift(1).rolling(WINDOW, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)

print(team_view.head(10))
print()
print("Teams with fewest matches in the window (check for sparse data):")
print(team_view["team"].value_counts().tail(10))

        date         team  goals_for  goals_against  \
0 2023-06-13  Afghanistan        1.0            6.0   
1 2023-09-03  Afghanistan        0.0            0.0   
2 2023-09-07  Afghanistan        1.0            1.0   
3 2023-09-12  Afghanistan        1.0            2.0   
4 2023-10-12  Afghanistan        1.0            0.0   
5 2023-10-17  Afghanistan        1.0            0.0   
6 2023-11-16  Afghanistan        1.0            8.0   
7 2023-11-21  Afghanistan        0.0            4.0   
8 2024-03-21  Afghanistan        0.0            0.0   
9 2024-03-26  Afghanistan        2.0            1.0   

                     tournament  rolling_goals_for  rolling_goals_against  
0              CAFA Nations Cup                NaN                    NaN  
1                      Friendly           1.000000               6.000000  
2                      Friendly           0.500000               3.000000  
3                      Friendly           0.666667               2.333333  
4  FIFA World 

## 6. Data cleaning: identify non-FIFA entities

The raw dataset includes CONIFA-style regional/stateless teams that can never appear in an actual World Cup.

In [7]:
# Get the full list of unique teams in the training data
all_teams = sorted(team_view["team"].unique())
print(f"Total unique teams/entities: {len(all_teams)}")
print(all_teams)

Total unique teams/entities: 252
['Afghanistan', 'Albania', 'Algeria', 'American Samoa', 'Andorra', 'Angola', 'Anguilla', 'Antigua and Barbuda', 'Argentina', 'Armenia', 'Aruba', 'Australia', 'Austria', 'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barbados', 'Basque Country', 'Belarus', 'Belgium', 'Belize', 'Benin', 'Bermuda', 'Bhutan', 'Bolivia', 'Bonaire', 'Bosnia and Herzegovina', 'Botswana', 'Brazil', 'British Virgin Islands', 'Brunei', 'Bulgaria', 'Burkina Faso', 'Burundi', 'Cambodia', 'Cameroon', 'Canada', 'Cape Verde', 'Cayman Islands', 'Central African Republic', 'Chad', 'Chameria', 'Chile', 'China', 'Colombia', 'Comoros', 'Congo', 'Cook Islands', 'Costa Rica', 'Croatia', 'Cuba', 'Curaçao', 'Cyprus', 'Czech Republic', 'DR Congo', 'Denmark', 'Djibouti', 'Dominica', 'Dominican Republic', 'East Turkestan', 'Ecuador', 'Egypt', 'El Salvador', 'England', 'Equatorial Guinea', 'Estonia', 'Eswatini', 'Ethiopia', 'Falkland Islands', 'Faroe Islands', 'Fiji', 'Finland', 'France', 'Fre

## 7. Remove non-FIFA entities from the training data

In [8]:
# These are CONIFA / non-FIFA entities present in the dataset: regions, stateless
# nations, and disputed territories that play international-style football but
# are NOT eligible for FIFA World Cup qualification. We exclude them since our
# model is specifically a World Cup predictor.
#
# NOTE: this list was built by manually reviewing the 252 unique team names in
# the dataset and flagging anything not a current FIFA member association.
# Worth re-checking if the dataset is updated or teams change status.

NON_FIFA_ENTITIES = [
    "Basque Country", "Chameria", "East Turkestan", "Frøya", "Galicia",
    "Gozo", "Guernsey", "Hitra", "Hmong", "Isle of Man", "Isle of Wight",
    "Jersey", "Kernow", "Menorca", "Orkney", "Raetia", "Shetland",
    "Székely Land", "Sápmi", "Tamil Eelam", "Tibet", "Ticino",
    "Two Sicilies", "Western Isles", "Ynys Môn", "Åland Islands", "Zanzibar"
]

# Filter completed matches: keep only matches where BOTH teams are FIFA members
completed_fifa = completed[
    (~completed["home_team"].isin(NON_FIFA_ENTITIES)) &
    (~completed["away_team"].isin(NON_FIFA_ENTITIES))
].copy()

print("Before filter:", completed.shape)
print("After filter:", completed_fifa.shape)
print("Rows removed:", completed.shape[0] - completed_fifa.shape[0])

Before filter: (3688, 9)
After filter: (3623, 9)
Rows removed: 65


## 8. Sanity check: confirm no real World Cup team was accidentally filtered out

In [9]:
# Sanity check: confirm all teams from our upcoming fixtures (the actual
# Round of 16 teams) survived the FIFA-only filter. If any of these are
# missing, it means we accidentally filtered out a real World Cup team,
# which would be a serious bug.

r16_teams = set(upcoming_fixtures["home_team"]).union(set(upcoming_fixtures["away_team"]))
teams_in_filtered_data = set(completed_fifa["home_team"]).union(set(completed_fifa["away_team"]))

missing_teams = r16_teams - teams_in_filtered_data
print("R16 teams missing from filtered data:", missing_teams if missing_teams else "None — all good")

R16 teams missing from filtered data: None — all good


## 9. Rebuild the team-match view on the cleaned, FIFA-only data

In [10]:
team_view = build_team_match_view(completed_fifa)

WINDOW = 5

team_view["rolling_goals_for"] = (
    team_view.groupby("team")["goals_for"]
    .apply(lambda x: x.shift(1).rolling(WINDOW, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)

team_view["rolling_goals_against"] = (
    team_view.groupby("team")["goals_against"]
    .apply(lambda x: x.shift(1).rolling(WINDOW, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)

print(team_view.shape)
print(team_view["team"].nunique(), "unique teams remaining")

(7246, 7)
223 unique teams remaining


## 10. Merge rolling form back onto the match-level data

In [11]:
# Merge rolling form back onto the match-level data.
#
# Right now, rolling_goals_for / rolling_goals_against live in team_view,
# which has ONE ROW PER TEAM PER MATCH (so each match appears twice: once
# from the home team's perspective, once from the away team's perspective).
#
# We need to get back to ONE ROW PER MATCH, but with both teams' form
# sitting side by side as separate columns (home team's form + away team's
# form), since that's the actual shape a model needs: given team A's form
# and team B's form, predict the outcome.
#
# The safe way to do this merge is on (date, team) as the join key — NOT
# just team name alone, because a team can appear many times across the
# dataset and we need the form value that was calculated for THIS specific
# match, not some other match involving the same team.

# Prepare a slim version of team_view with just the columns we need to merge in
form_lookup = team_view[["date", "team", "rolling_goals_for", "rolling_goals_against"]].copy()

# Merge in the HOME team's form
# We match team_view's (date, team) against completed_fifa's (date, home_team)
completed_features = completed_fifa.merge(
    form_lookup,
    left_on=["date", "home_team"],
    right_on=["date", "team"],
    how="left"
)
completed_features = completed_features.rename(columns={
    "rolling_goals_for": "home_rolling_goals_for",
    "rolling_goals_against": "home_rolling_goals_against"
})
completed_features = completed_features.drop(columns=["team"])  # drop the redundant join key column

# Merge in the AWAY team's form, same logic
completed_features = completed_features.merge(
    form_lookup,
    left_on=["date", "away_team"],
    right_on=["date", "team"],
    how="left"
)
completed_features = completed_features.rename(columns={
    "rolling_goals_for": "away_rolling_goals_for",
    "rolling_goals_against": "away_rolling_goals_against"
})
completed_features = completed_features.drop(columns=["team"])

# Sanity checks after the merge:
# 1. Row count should be unchanged (a left merge on date+team should be 1:1,
#    since a team plays at most once per date in this dataset)
print("Before merge:", completed_fifa.shape[0], "rows")
print("After merge:", completed_features.shape[0], "rows")

# 2. Check for unexpected duplication (if row count grew, something's matching
#    more than once per key, which would mean duplicate (date, team) rows exist
#    somewhere and needs investigating)
assert completed_features.shape[0] == completed_fifa.shape[0], \
    "Row count changed after merge — check for duplicate (date, team) pairs"

# 3. Preview the result
print(completed_features[[
    "date", "home_team", "away_team", "home_score", "away_score",
    "home_rolling_goals_for", "home_rolling_goals_against",
    "away_rolling_goals_for", "away_rolling_goals_against"
]].head(10))

Before merge: 3623 rows
After merge: 3635 rows


AssertionError: Row count changed after merge — check for duplicate (date, team) pairs

## 11. Investigate a row-count mismatch after the merge

The merge produced more rows than expected — investigating duplicate (date, team) pairs.

In [12]:
# Find duplicate (date, team) pairs in team_view — these are cases where the
# same team appears more than once on the same date. This could be:
#   (a) a genuine data quirk (e.g. two mini-tournament matches erroneously
#       logged on the same date), or
#   (b) something structural we're missing (e.g. a match that got entered twice)

dupes = team_view[team_view.duplicated(subset=["date", "team"], keep=False)]
dupes_sorted = dupes.sort_values(["team", "date"])
print(f"Found {len(dupes_sorted)} rows involved in duplicate (date, team) pairs")
print(dupes_sorted[["date", "team", "goals_for", "goals_against", "tournament"]].to_string(index=False))

Found 10 rows involved in duplicate (date, team) pairs
      date                  team  goals_for  goals_against                                        tournament
2026-06-06        Cayman Islands        1.0            4.0                                          Friendly
2026-06-06        Cayman Islands        1.0            4.0                                          Friendly
2024-09-02                  Fiji        1.0            0.0                                Tri-Nations Series
2024-09-02                  Fiji        1.0            1.0                                Tri-Nations Series
2026-06-06             Gibraltar        4.0            1.0                                          Friendly
2026-06-06             Gibraltar        4.0            1.0                                          Friendly
2026-06-01              Maldives        0.0            1.0 Diamond Jubilee International Football Tournament
2026-06-01              Maldives        0.0            3.0 Diamond Jubile

## 12. Fix: use a stable match ID as the join key instead of (date, team)

Same-day double-headers exist for some lower-tier teams, so (date, team) isn't a reliable merge key.

In [13]:
# Give completed_fifa a stable unique match ID up front. This becomes the
# real join key going forward instead of (date, team), which we just proved
# isn't reliably unique (same-day double-headers exist for some teams).
completed_fifa = completed_fifa.reset_index(drop=True)
completed_fifa["match_id"] = completed_fifa.index

# Rebuild team_view carrying match_id through, so each team-perspective row
# still traces back to exactly one specific match.
def build_team_match_view(df):
    home_view = df[["match_id", "date", "home_team", "home_score", "away_score", "tournament"]].copy()
    home_view.columns = ["match_id", "date", "team", "goals_for", "goals_against", "tournament"]

    away_view = df[["match_id", "date", "away_team", "away_score", "home_score", "tournament"]].copy()
    away_view.columns = ["match_id", "date", "team", "goals_for", "goals_against", "tournament"]

    team_view = pd.concat([home_view, away_view], ignore_index=True)
    team_view = team_view.sort_values(["team", "date"]).reset_index(drop=True)
    return team_view

team_view = build_team_match_view(completed_fifa)

# Recompute rolling form exactly as before — this part doesn't change,
# since it only depends on team + date ordering, not match_id
WINDOW = 5
team_view["rolling_goals_for"] = (
    team_view.groupby("team")["goals_for"]
    .apply(lambda x: x.shift(1).rolling(WINDOW, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)
team_view["rolling_goals_against"] = (
    team_view.groupby("team")["goals_against"]
    .apply(lambda x: x.shift(1).rolling(WINDOW, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)

# Now merge on match_id + team — guaranteed unique per (match_id, team) pair,
# since match_id alone identifies one specific match, and each match only
# has one home_team and one away_team.
form_lookup = team_view[["match_id", "team", "rolling_goals_for", "rolling_goals_against"]].copy()

completed_features = completed_fifa.merge(
    form_lookup, left_on=["match_id", "home_team"], right_on=["match_id", "team"], how="left"
).rename(columns={
    "rolling_goals_for": "home_rolling_goals_for",
    "rolling_goals_against": "home_rolling_goals_against"
}).drop(columns=["team"])

completed_features = completed_features.merge(
    form_lookup, left_on=["match_id", "away_team"], right_on=["match_id", "team"], how="left"
).rename(columns={
    "rolling_goals_for": "away_rolling_goals_for",
    "rolling_goals_against": "away_rolling_goals_against"
}).drop(columns=["team"])

print("Before merge:", completed_fifa.shape[0], "rows")
print("After merge:", completed_features.shape[0], "rows")

assert completed_features.shape[0] == completed_fifa.shape[0], \
    "Row count still doesn't match — needs further investigation"

Before merge: 3623 rows
After merge: 3623 rows


## 13. Remove exact-duplicate match rows (data entry errors)

In [14]:
# Remove exact duplicate match rows (same date, same teams, same score —
# i.e. the same match logged twice in the source data). Keep the first
# occurrence, drop the rest.
before = completed_features.shape[0]
completed_features = completed_features.drop_duplicates(
    subset=["date", "home_team", "away_team", "home_score", "away_score"],
    keep="first"
).reset_index(drop=True)
after = completed_features.shape[0]

print(f"Removed {before - after} exact duplicate match rows")
print("Final shape:", completed_features.shape)

Removed 1 exact duplicate match rows
Final shape: (3622, 14)


## 14. Preview the merged feature set

In [15]:
print(completed_features[[
    "date", "home_team", "away_team", "home_score", "away_score",
    "home_rolling_goals_for", "home_rolling_goals_against",
    "away_rolling_goals_for", "away_rolling_goals_against"
]].head(10))

        date    home_team             away_team  home_score  away_score  \
0 2023-01-02     Thailand              Cambodia         3.0         1.0   
1 2023-01-02  Philippines             Indonesia         1.0         2.0   
2 2023-01-03      Vietnam               Myanmar         3.0         0.0   
3 2023-01-03     Malaysia             Singapore         4.0         1.0   
4 2023-01-06         Iraq                  Oman         0.0         0.0   
5 2023-01-06        Yemen          Saudi Arabia         0.0         2.0   
6 2023-01-06    Indonesia               Vietnam         0.0         0.0   
7 2023-01-07      Bahrain  United Arab Emirates         2.0         1.0   
8 2023-01-07       Kuwait                 Qatar         0.0         2.0   
9 2023-01-07     Malaysia              Thailand         1.0         0.0   

   home_rolling_goals_for  home_rolling_goals_against  away_rolling_goals_for  \
0                     NaN                         NaN                     NaN   
1           

## 15. Check how much data is affected by missing (early-window) rolling form

In [16]:
nan_home = completed_features["home_rolling_goals_for"].isna().sum()
nan_away = completed_features["away_rolling_goals_for"].isna().sum()
print(f"Rows with missing home form: {nan_home} ({nan_home/len(completed_features)*100:.1f}%)")
print(f"Rows with missing away form: {nan_away} ({nan_away/len(completed_features)*100:.1f}%)")

Rows with missing home form: 103 (2.8%)
Rows with missing away form: 120 (3.3%)


## 16. Fill missing rolling form values with the league average

In [17]:
# Fill missing rolling-form values with the overall league average across
# all completed matches. This gives teams with no prior history yet (early
# in the 2023 window) a neutral, reasonable starting assumption rather than
# leaving a gap the model can't handle, or dropping otherwise-valid matches.

league_avg_goals_for = completed_features[["home_rolling_goals_for", "away_rolling_goals_for"]].stack().mean()
league_avg_goals_against = completed_features[["home_rolling_goals_against", "away_rolling_goals_against"]].stack().mean()

print(f"League average goals-for (rolling): {league_avg_goals_for:.3f}")
print(f"League average goals-against (rolling): {league_avg_goals_against:.3f}")

completed_features["home_rolling_goals_for"] = completed_features["home_rolling_goals_for"].fillna(league_avg_goals_for)
completed_features["home_rolling_goals_against"] = completed_features["home_rolling_goals_against"].fillna(league_avg_goals_against)
completed_features["away_rolling_goals_for"] = completed_features["away_rolling_goals_for"].fillna(league_avg_goals_for)
completed_features["away_rolling_goals_against"] = completed_features["away_rolling_goals_against"].fillna(league_avg_goals_against)

# Confirm no NaNs remain in these columns
print("Remaining NaNs:", completed_features[[
    "home_rolling_goals_for", "home_rolling_goals_against",
    "away_rolling_goals_for", "away_rolling_goals_against"
]].isna().sum().sum())

League average goals-for (rolling): 1.385
League average goals-against (rolling): 1.353
Remaining NaNs: 0


## 17. Feature engineering: head-to-head history

**Caveat:** median number of prior meetings per team pair in this window is 0 — this feature carries real signal only for frequently-paired regional rivals.

In [18]:
# Head-to-head history: before each match, how has this exact pairing of
# teams fared against each other historically (within our filtered window)?
#
# Why this matters: some team pairs have persistent patterns beyond what
# general form captures — tactical matchups, rivalries, psychological
# factors. E.g. a team might have a strong H2H record against a specific
# opponent even while both teams' general form is similar.
#
# Approach: for each match, look at all PRIOR matches (before this date)
# between these same two teams (regardless of which was home/away back
# then), and compute the home team's historical win rate against this
# specific opponent.

def compute_h2h_features(df):
    df = df.sort_values("date").reset_index(drop=True)
    h2h_win_rate = []
    h2h_matches_played = []

    # Track results team-pair by team-pair as we move through time in order
    history = {}  # key: frozenset({team_a, team_b}), value: list of (date, winner)

    for idx, row in df.iterrows():
        home, away = row["home_team"], row["away_team"]
        key = frozenset([home, away])

        past_matches = history.get(key, [])
        h2h_matches_played.append(len(past_matches))

        if len(past_matches) == 0:
            # No prior meetings between these two teams in our window
            h2h_win_rate.append(0.5)  # neutral prior: no data, assume even
        else:
            # Count how many of the past meetings the CURRENT home team won
            home_wins = sum(1 for winner in past_matches if winner == home)
            h2h_win_rate.append(home_wins / len(past_matches))

        # Now record this match's own result into history, for future lookups
        if row["home_score"] > row["away_score"]:
            winner = home
        elif row["away_score"] > row["home_score"]:
            winner = away
        else:
            winner = "draw"

        history.setdefault(key, []).append(winner)

    df["h2h_home_win_rate"] = h2h_win_rate
    df["h2h_matches_played"] = h2h_matches_played
    return df

completed_features = compute_h2h_features(completed_features)

print(completed_features[[
    "date", "home_team", "away_team", "home_score", "away_score",
    "h2h_home_win_rate", "h2h_matches_played"
]].head(10))

print()
print("Distribution of h2h_matches_played (how much history most pairs have):")
print(completed_features["h2h_matches_played"].describe())

        date    home_team             away_team  home_score  away_score  \
0 2023-01-02     Thailand              Cambodia         3.0         1.0   
1 2023-01-02  Philippines             Indonesia         1.0         2.0   
2 2023-01-03      Vietnam               Myanmar         3.0         0.0   
3 2023-01-03     Malaysia             Singapore         4.0         1.0   
4 2023-01-06         Iraq                  Oman         0.0         0.0   
5 2023-01-06        Yemen          Saudi Arabia         0.0         2.0   
6 2023-01-06    Indonesia               Vietnam         0.0         0.0   
7 2023-01-07      Bahrain  United Arab Emirates         2.0         1.0   
8 2023-01-07       Kuwait                 Qatar         0.0         2.0   
9 2023-01-07     Malaysia              Thailand         1.0         0.0   

   h2h_home_win_rate  h2h_matches_played  
0                0.5                   0  
1                0.5                   0  
2                0.5                   0  
3 

## 18. Inspect tournament types present in the data

In [19]:
# Before assigning weights, see what tournament types actually exist in
# our filtered data and how many matches fall into each — this tells us
# where to focus and whether any need special-casing.

print(completed_features["tournament"].value_counts())

tournament
Friendly                                             919
FIFA World Cup qualification                         901
CONCACAF Nations League                              249
African Cup of Nations qualification                 239
UEFA Euro qualification                              239
UEFA Nations League                                  190
African Cup of Nations                               104
FIFA World Cup                                        90
AFC Asian Cup qualification                           65
Gold Cup                                              62
FIFA Series                                           57
UEFA Euro                                             51
AFC Asian Cup                                         51
CONCACAF Series                                       35
Copa América                                          32
Arab Cup                                              31
Gulf Cup                                              30
ASEAN Championship  

## 19. Feature engineering: tournament competitiveness weighting

A judgment-call 4-tier weighting scheme (World Cup finals > qualifiers > regional leagues > friendlies), used to scale each match's influence when fitting team strength.

In [20]:
# Tournament competitiveness weighting.
#
# Not every match should count equally when we later fit team-strength
# ratings — a World Cup win says a lot more about a team's real strength
# than a friendly win. This weight will be used to scale how much each
# match "moves" a team's rating (bigger weight = bigger impact).
#
# Tiers, roughly by stakes/quality of opposition:
#   1.0  = major tournament finals (World Cup, continental championships)
#   0.85 = major tournament qualifiers (still high stakes, slightly lower quality)
#   0.7  = regional nations leagues / cups (competitive, but lower overall level)
#   0.5  = friendlies and everything else not explicitly listed
#
# This is a first-pass, editable scheme — the exact numbers are a modeling
# choice, not a hard fact, and can be tuned later based on backtest results.

TOURNAMENT_WEIGHTS = {
    # Tier 1: major tournament finals
    "FIFA World Cup": 1.0,
    "UEFA Euro": 1.0,
    "Copa América": 1.0,
    "African Cup of Nations": 1.0,
    "AFC Asian Cup": 1.0,
    "Gold Cup": 1.0,

    # Tier 2: major tournament qualification
    "FIFA World Cup qualification": 0.85,
    "UEFA Euro qualification": 0.85,
    "African Cup of Nations qualification": 0.85,
    "AFC Asian Cup qualification": 0.85,
    "Gold Cup qualification": 0.85,

    # Tier 3: regional nations leagues / cups (competitive, lower overall level)
    "UEFA Nations League": 0.7,
    "CONCACAF Nations League": 0.7,
    "CONCACAF Series": 0.7,
    "Arab Cup": 0.7,
    "Gulf Cup": 0.7,
    "ASEAN Championship": 0.7,
    "COSAFA Cup": 0.7,
    "SAFF Cup": 0.7,
    "Oceania Nations Cup": 0.7,
}

# Anything not explicitly listed (Friendly, FIFA Series, smaller regional
# tournaments, etc.) defaults to 0.5 — treated as lower-stakes.
DEFAULT_WEIGHT = 0.5

completed_features["tournament_weight"] = (
    completed_features["tournament"].map(TOURNAMENT_WEIGHTS).fillna(DEFAULT_WEIGHT)
)

# Check the result: how many matches fall into each weight tier
print(completed_features["tournament_weight"].value_counts().sort_index(ascending=False))

tournament_weight
1.00     390
0.85    1467
0.70     611
0.50    1154
Name: count, dtype: int64


## 20. The model: Poisson attack/defense framework

Each team gets an attack strength and defense strength, estimated relative to a fixed reference team (Mexico), via maximum likelihood.

In [21]:
# Poisson attack/defense model — the core idea:
#
# Every team has two hidden strength parameters:
#   - attack_strength:  how many goals they tend to score, relative to league average
#   - defense_strength: how many goals they tend to concede, relative to league average
#     (lower defense_strength = better defense, since it multiplies into opponent's
#      expected goals)
#
# For a given match, we estimate:
#   home_expected_goals = league_avg_goals * home_team.attack_strength
#                          * away_team.defense_strength * home_advantage
#   away_expected_goals = league_avg_goals * away_team.attack_strength
#                          * home_team.defense_strength
#
# These expected goals then feed a Poisson distribution to get the full
# scoreline probability grid (P(0-0), P(1-0), P(2-1), etc.)
#
# We estimate attack_strength and defense_strength per team by maximizing
# the likelihood of the actual observed scores in our training data —
# i.e. finding the parameters that make the real results "most probable"
# under the model. This is standard practice for this kind of model
# (originally popularized by Dixon & Coles, 1997).

import numpy as np
from scipy.optimize import minimize
from scipy.stats import poisson

teams = sorted(set(completed_features["home_team"]).union(set(completed_features["away_team"])))
team_to_idx = {team: i for i, team in enumerate(teams)}
n_teams = len(teams)

print(f"Number of teams to estimate parameters for: {n_teams}")

Number of teams to estimate parameters for: 223


## 21. Build the log-likelihood function

Includes the identifiability fix (fixing one reference team's strength at 1.0) needed for the optimization to have a unique solution.

In [22]:
# Building the log-likelihood function for the Poisson attack/defense model.
#
# IMPORTANT — identifiability problem:
# If every team's attack_strength was doubled and every defense_strength was
# halved, the predicted expected goals would stay exactly the same (they
# multiply together). This means there are infinitely many parameter
# combinations that fit the data equally well — the model is "unidentified"
# unless we pin something down.
#
# Standard fix: fix ONE team as the reference, with attack_strength = 1.0
# and defense_strength = 1.0 by definition. Every other team's strength is
# then estimated RELATIVE to this reference team. The choice of reference
# team doesn't matter for predictions — it's just an anchor point.
#
# We pick the team with the most matches in our data as the reference,
# since a well-estimated anchor is more reliable than a team with only 1-2
# matches.

match_counts = pd.concat([completed_features["home_team"], completed_features["away_team"]]).value_counts()
reference_team = match_counts.idxmax()
print(f"Reference team (anchor, attack=defense=1.0 by definition): {reference_team} ({match_counts.max()} matches)")

reference_idx = team_to_idx[reference_team]

# Prepare match data as plain numpy arrays for fast vectorized computation
home_idx_arr = completed_features["home_team"].map(team_to_idx).values
away_idx_arr = completed_features["away_team"].map(team_to_idx).values
home_score_arr = completed_features["home_score"].values
away_score_arr = completed_features["away_score"].values
weight_arr = completed_features["tournament_weight"].values

# Parameters to estimate (all in LOG space, so they stay positive after
# exponentiating — a common trick to avoid the optimizer wandering into
# invalid negative "strength" values):
#   - log_attack[team]   for every team except the reference (fixed at 0 = exp(0) = 1.0)
#   - log_defense[team]  for every team except the reference (fixed at 0 = exp(0) = 1.0)
#   - log_home_advantage (a single global multiplier for playing at home)
#   - log_avg_goals      (the baseline league-average expected goals per team per match)

n_free_teams = n_teams - 1  # all teams except the reference

def unpack_params(params):
    log_attack_free = params[0:n_free_teams]
    log_defense_free = params[n_free_teams:2*n_free_teams]
    log_home_advantage = params[-2]
    log_avg_goals = params[-1]

    # Rebuild full-length arrays (length n_teams), inserting the fixed
    # reference team's value (0.0 in log space = 1.0 actual strength)
    log_attack = np.insert(log_attack_free, reference_idx, 0.0)
    log_defense = np.insert(log_defense_free, reference_idx, 0.0)

    return log_attack, log_defense, log_home_advantage, log_avg_goals

def negative_log_likelihood(params):
    log_attack, log_defense, log_home_advantage, log_avg_goals = unpack_params(params)

    avg_goals = np.exp(log_avg_goals)
    home_advantage = np.exp(log_home_advantage)

    # Expected goals for each match, vectorized across all matches at once
    home_attack = np.exp(log_attack[home_idx_arr])
    home_defense = np.exp(log_defense[home_idx_arr])
    away_attack = np.exp(log_attack[away_idx_arr])
    away_defense = np.exp(log_defense[away_idx_arr])

    home_expected = avg_goals * home_attack * away_defense * home_advantage
    away_expected = avg_goals * away_attack * home_defense

    # Poisson log-likelihood per match: k*log(lambda) - lambda
    # (the log(k!) term is dropped since it's constant w.r.t. parameters
    # and doesn't affect where the minimum is)
    ll_home = home_score_arr * np.log(home_expected) - home_expected
    ll_away = away_score_arr * np.log(away_expected) - away_expected

    # Weight each match's contribution by its tournament competitiveness
    # weight, so a World Cup match's result influences the fit more than
    # a friendly's
    weighted_ll = weight_arr * (ll_home + ll_away)

    return -np.sum(weighted_ll)  # negative, since scipy MINIMIZES

# Initial guess: everyone starts equal (log 0 = strength 1.0), average
# goals guessed from the data's actual average, no home advantage yet
initial_params = np.zeros(2 * n_free_teams + 2)
initial_params[-1] = np.log(completed_features["home_score"].mean())  # rough starting point for avg_goals

print("Set up complete. Number of parameters to estimate:", len(initial_params))

Reference team (anchor, attack=defense=1.0 by definition): Mexico (61 matches)
Set up complete. Number of parameters to estimate: 446


## 22. Fit the model (first pass)

In [23]:
# Run the optimization: find attack/defense strengths + home advantage +
# average goals that make our observed match results as likely as possible
# under the Poisson model.
#
# Method choice: 'L-BFGS-B' is a standard, efficient choice for this kind
# of smooth, unconstrained (well, box-unconstrained here) optimization
# problem with many parameters. It uses gradient information estimated
# numerically to converge faster than simpler methods.

result = minimize(
    negative_log_likelihood,
    initial_params,
    method="L-BFGS-B",
    options={"maxiter": 1000, "disp": True}
)

print("Optimization converged:", result.success)
print("Final negative log-likelihood:", result.fun)

# Unpack the fitted parameters back into readable team strengths
log_attack, log_defense, log_home_advantage, log_avg_goals = unpack_params(result.x)

attack_strength = np.exp(log_attack)
defense_strength = np.exp(log_defense)
home_advantage = np.exp(log_home_advantage)
avg_goals = np.exp(log_avg_goals)

print()
print(f"Fitted home advantage multiplier: {home_advantage:.3f}")
print(f"Fitted league-average goals per team per match: {avg_goals:.3f}")

# Build a readable team-strength table
team_strength = pd.DataFrame({
    "team": teams,
    "attack_strength": attack_strength,
    "defense_strength": defense_strength
}).sort_values("attack_strength", ascending=False)

print()
print("Top 15 attacking teams (relative to Mexico = 1.0):")
print(team_strength.head(15).to_string(index=False))

print()
print("Bottom 15 defenses (higher = worse defense, relative to Mexico = 1.0):")
print(team_strength.sort_values("defense_strength", ascending=False).head(15).to_string(index=False))

C:\Users\Waconia\AppData\Local\Temp\ipykernel_3500\1433242899.py:10: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  result = minimize(


Optimization converged: False
Final negative log-likelihood: 3479.908870755782

Fitted home advantage multiplier: 1.259
Fitted league-average goals per team per match: 0.986

Top 15 attacking teams (relative to Mexico = 1.0):
       team  attack_strength  defense_strength
      Spain         3.752980          0.310087
     Norway         3.536058          0.558956
    Germany         3.418217          0.520242
     France         3.336383          0.300222
   Portugal         3.276973          0.374861
Netherlands         3.264619          0.495301
     Brazil         3.230505          0.401427
  Argentina         3.123615          0.187766
      Japan         3.089704          0.440033
   Colombia         3.028925          0.320818
    Belgium         2.854233          0.421051
    England         2.836558          0.311551
Switzerland         2.777758          0.467167
    Austria         2.586766          0.460589
    Senegal         2.561480          0.402415

Bottom 15 defenses (h

## 23. Continue fitting: warm-start from the previous result

The first optimization run did not fully converge within its iteration limit — refitting from where it left off.

In [24]:
# Try again with more iterations and a tighter tolerance, restarting from
# where the previous optimization left off (result.x) rather than from
# scratch — this "warm start" usually converges much faster than starting
# from all-zeros again.

result2 = minimize(
    negative_log_likelihood,
    result.x,  # warm start from previous result instead of initial_params
    method="L-BFGS-B",
    options={"maxiter": 5000, "maxfun": 20000}
)

print("Optimization converged:", result2.success)
print("Final negative log-likelihood:", result2.fun)
print("Previous negative log-likelihood:", result.fun)

Optimization converged: False
Final negative log-likelihood: 3448.3032321744186
Previous negative log-likelihood: 3479.908870755782


## 24. Continue fitting again

In [25]:
# Keep warm-starting from the previous result until the improvement
# becomes negligible (a sign we've actually reached the optimum, not
# just run out of iterations again)

result3 = minimize(
    negative_log_likelihood,
    result2.x,
    method="L-BFGS-B",
    options={"maxiter": 5000, "maxfun": 20000}
)

print("Optimization converged:", result3.success)
print("Final negative log-likelihood:", result3.fun)
print("Previous negative log-likelihood:", result2.fun)
print("Improvement:", result2.fun - result3.fun)

Optimization converged: False
Final negative log-likelihood: 3437.68406427886
Previous negative log-likelihood: 3448.3032321744186
Improvement: 10.619167895558348


## 25. Automate the warm-start loop until improvement is negligible

In [26]:
# Automated warm-start loop: keep refitting until the improvement per
# round becomes negligible, or we hit a safety cap on rounds (to avoid
# an infinite loop if something's oscillating instead of converging)

current_result = result3
for round_num in range(10):
    next_result = minimize(
        negative_log_likelihood,
        current_result.x,
        method="L-BFGS-B",
        options={"maxiter": 5000, "maxfun": 20000}
    )
    improvement = current_result.fun - next_result.fun
    print(f"Round {round_num+1}: NLL = {next_result.fun:.4f}, improvement = {improvement:.6f}, converged = {next_result.success}")

    current_result = next_result

    if next_result.success or improvement < 0.001:
        print("Stopping — converged or improvement negligible.")
        break

final_result = current_result

Round 1: NLL = 3432.9472, improvement = 4.736841, converged = False
Round 2: NLL = 3428.5213, improvement = 4.425954, converged = False
Round 3: NLL = 3427.1162, improvement = 1.405026, converged = False
Round 4: NLL = 3426.7406, improvement = 0.375627, converged = False
Round 5: NLL = 3426.5623, improvement = 0.178304, converged = False
Round 6: NLL = 3426.4402, improvement = 0.122126, converged = False
Round 7: NLL = 3426.3938, improvement = 0.046407, converged = False
Round 8: NLL = 3426.3565, improvement = 0.037283, converged = False
Round 9: NLL = 3426.3152, improvement = 0.041321, converged = False
Round 10: NLL = 3426.2982, improvement = 0.016955, converged = False


## 26. Extract final fitted parameters: attack/defense strengths, home advantage, average goals

In [27]:
final_result = current_result
log_attack, log_defense, log_home_advantage, log_avg_goals = unpack_params(final_result.x)
attack_strength = np.exp(log_attack)
defense_strength = np.exp(log_defense)
home_advantage = np.exp(log_home_advantage)
avg_goals = np.exp(log_avg_goals)
print(f"Home advantage: {home_advantage:.3f}, avg goals: {avg_goals:.3f}")

Home advantage: 1.255, avg goals: 0.831


## 27. Generate predictions for the 7 live Round of 16 fixtures

In [28]:
# Generate predictions for the 7 upcoming Round of 16 fixtures using our
# fitted attack/defense strengths.
#
# For each match:
#   1. Look up both teams' attack/defense strength (falling back to
#      average strength = 1.0 if a team is somehow missing, though none
#      should be given our earlier R16 sanity check)
#   2. Compute expected goals for each side
#   3. Build the full scoreline probability grid using independent
#      Poisson distributions for home and away goals
#   4. Sum the grid to get overall Win / Draw / Loss probabilities

team_strength_lookup = pd.DataFrame({
    "team": teams,
    "attack": attack_strength,
    "defense": defense_strength
}).set_index("team")

MAX_GOALS = 8  # cap the scoreline grid — probability beyond 8 goals is negligible

def predict_match(home_team, away_team, neutral):
    home_attack = team_strength_lookup.loc[home_team, "attack"]
    home_defense = team_strength_lookup.loc[home_team, "defense"]
    away_attack = team_strength_lookup.loc[away_team, "attack"]
    away_defense = team_strength_lookup.loc[away_team, "defense"]

    # Only apply home advantage if the match isn't at a neutral venue —
    # World Cup knockout matches are almost always neutral, so this will
    # typically switch the boost off, which matches football reality
    # (no true "home" team in a World Cup final at a neutral stadium)
    effective_home_advantage = 1.0 if neutral else home_advantage

    home_expected = avg_goals * home_attack * away_defense * effective_home_advantage
    away_expected = avg_goals * away_attack * home_defense

    # Build scoreline probability grid
    home_probs = poisson.pmf(np.arange(MAX_GOALS+1), home_expected)
    away_probs = poisson.pmf(np.arange(MAX_GOALS+1), away_expected)
    grid = np.outer(home_probs, away_probs)

    home_win_prob = np.sum(np.tril(grid, -1))   # home_score > away_score
    draw_prob = np.sum(np.diag(grid))            # home_score == away_score
    away_win_prob = np.sum(np.triu(grid, 1))     # away_score > home_score

    return home_win_prob, draw_prob, away_win_prob, home_expected, away_expected

# Run predictions for all 7 upcoming fixtures
print(f"{'Match':<30} {'Home Win':>9} {'Draw':>7} {'Away Win':>9}   Expected Score")
for _, row in upcoming_fixtures.iterrows():
    home, away, neutral = row["home_team"], row["away_team"], row["neutral"]
    hw, d, aw, he, ae = predict_match(home, away, neutral)
    matchup = f"{home} vs {away}"
    print(f"{matchup:<30} {hw*100:>8.1f}% {d*100:>6.1f}% {aw*100:>8.1f}%   {he:.2f} - {ae:.2f}")

Match                           Home Win    Draw  Away Win   Expected Score
Brazil vs Norway                   43.0%   23.5%     33.5%   1.69 - 1.46
Mexico vs England                  21.0%   29.8%     49.2%   0.70 - 1.24
Portugal vs Spain                  25.4%   27.2%     47.4%   0.94 - 1.40
United States vs Belgium           23.3%   22.7%     53.9%   1.14 - 1.85
Argentina vs Egypt                 66.2%   24.8%      9.0%   1.53 - 0.39
Switzerland vs Colombia            26.5%   27.5%     46.0%   0.96 - 1.36
France vs Morocco                  40.4%   35.0%     24.5%   0.93 - 0.65


## 28. Refinement: Dixon-Coles correction for low-scoring matches

Plain independent Poisson underestimates low-scoring draws (0-0, 1-1). Fitting a correlation parameter (rho) to correct this.

In [29]:
# Dixon-Coles correction for low-scoring matches.
#
# The problem: plain independent Poisson (what we just used) systematically
# gets 0-0, 1-0, 0-1, and 1-1 scorelines slightly wrong — because in real
# football, these low scores are correlated in ways a simple product of two
# independent Poisson distributions doesn't capture (e.g. teams sitting back
# to protect a 1-0 lead make 1-1 slightly less likely than pure chance would
# suggest, and 0-0 slightly more likely due to cautious tactics in tight games).
#
# Dixon & Coles (1997) fixed this with a small correction factor, tau, applied
# ONLY to the four low-scoring cells (0-0, 1-0, 0-1, 1-1). Every other
# scoreline is left as plain Poisson.
#
# tau depends on a single extra parameter, rho, which we estimate from the
# data (typically a small negative number, around -0.1 to -0.2 for real
# football data).

def dixon_coles_tau(home_goals, away_goals, home_expected, away_expected, rho):
    if home_goals == 0 and away_goals == 0:
        return 1 - (home_expected * away_expected * rho)
    elif home_goals == 0 and away_goals == 1:
        return 1 + (home_expected * rho)
    elif home_goals == 1 and away_goals == 0:
        return 1 + (away_expected * rho)
    elif home_goals == 1 and away_goals == 1:
        return 1 - rho
    else:
        return 1.0  # no correction outside the four low-scoring cells

# We estimate rho by maximizing likelihood again, same idea as before, but
# this time holding attack/defense/home_advantage/avg_goals FIXED at what
# we already fitted, and only solving for this one new parameter. This is
# a simplification (a fully joint refit would be more "correct" but much
# slower) — reasonable for a v1.

def negative_log_likelihood_rho(rho_param):
    rho = rho_param[0]
    home_attack_arr = attack_strength[home_idx_arr]
    home_defense_arr = defense_strength[home_idx_arr]
    away_attack_arr = attack_strength[away_idx_arr]
    away_defense_arr = defense_strength[away_idx_arr]

    neutral_mask = completed_features["neutral"].values
    effective_home_adv = np.where(neutral_mask, 1.0, home_advantage)

    home_expected_arr = avg_goals * home_attack_arr * away_defense_arr * effective_home_adv
    away_expected_arr = avg_goals * away_attack_arr * home_defense_arr

    total_ll = 0.0
    for i in range(len(home_score_arr)):
        hg, ag = int(home_score_arr[i]), int(away_score_arr[i])
        he, ae = home_expected_arr[i], away_expected_arr[i]

        base_prob = poisson.pmf(hg, he) * poisson.pmf(ag, ae)
        tau = dixon_coles_tau(hg, ag, he, ae, rho)

        # Guard against tau going negative or zero (invalid probability) —
        # can happen with an extreme rho during optimization search
        prob = max(base_prob * tau, 1e-10)
        total_ll += weight_arr[i] * np.log(prob)

    return -total_ll

rho_result = minimize(
    negative_log_likelihood_rho,
    x0=[-0.1],  # reasonable starting guess based on typical published values
    method="Nelder-Mead"
)

fitted_rho = rho_result.x[0]
print(f"Fitted rho: {fitted_rho:.4f}")
print(f"Converged: {rho_result.success}")

Fitted rho: -0.0892
Converged: True


## 29. Rebuild predictions with the Dixon-Coles correction applied

In [30]:
# Rebuild the prediction function with the Dixon-Coles correction applied
# to the four low-scoring cells (0-0, 1-0, 0-1, 1-1), leaving every other
# scoreline as plain Poisson.

def predict_match_dc(home_team, away_team, neutral, rho):
    home_attack = team_strength_lookup.loc[home_team, "attack"]
    home_defense = team_strength_lookup.loc[home_team, "defense"]
    away_attack = team_strength_lookup.loc[away_team, "attack"]
    away_defense = team_strength_lookup.loc[away_team, "defense"]

    effective_home_advantage = 1.0 if neutral else home_advantage

    home_expected = avg_goals * home_attack * away_defense * effective_home_advantage
    away_expected = avg_goals * away_attack * home_defense

    home_probs = poisson.pmf(np.arange(MAX_GOALS+1), home_expected)
    away_probs = poisson.pmf(np.arange(MAX_GOALS+1), away_expected)
    grid = np.outer(home_probs, away_probs)

    # Apply the Dixon-Coles correction to the four low-scoring cells only
    grid[0, 0] *= dixon_coles_tau(0, 0, home_expected, away_expected, rho)
    grid[1, 0] *= dixon_coles_tau(1, 0, home_expected, away_expected, rho)
    grid[0, 1] *= dixon_coles_tau(0, 1, home_expected, away_expected, rho)
    grid[1, 1] *= dixon_coles_tau(1, 1, home_expected, away_expected, rho)

    # Renormalize so probabilities still sum to 1 after the correction
    # (the tau adjustments slightly shift total probability mass, so we
    # rescale the whole grid back to sum to exactly 1.0)
    grid = grid / grid.sum()

    home_win_prob = np.sum(np.tril(grid, -1))
    draw_prob = np.sum(np.diag(grid))
    away_win_prob = np.sum(np.triu(grid, 1))

    return home_win_prob, draw_prob, away_win_prob, home_expected, away_expected

print(f"{'Match':<30} {'Home Win':>9} {'Draw':>7} {'Away Win':>9}   (was Draw before DC)")
for _, row in upcoming_fixtures.iterrows():
    home, away, neutral = row["home_team"], row["away_team"], row["neutral"]
    hw, d, aw, he, ae = predict_match_dc(home, away, neutral, fitted_rho)
    matchup = f"{home} vs {away}"
    print(f"{matchup:<30} {hw*100:>8.1f}% {d*100:>6.1f}% {aw*100:>8.1f}%")

Match                           Home Win    Draw  Away Win   (was Draw before DC)
Brazil vs Norway                   42.1%   25.4%     32.5%
Mexico vs England                  19.8%   32.1%     48.1%
Portugal vs Spain                  24.3%   29.4%     46.3%
United States vs Belgium           22.4%   24.6%     53.0%
Argentina vs Egypt                 65.4%   26.3%      8.2%
Switzerland vs Colombia            25.3%   29.8%     44.9%
France vs Morocco                  39.3%   37.3%     23.4%


## 30. Backtest: how good is this model, honestly?

**Important caveat:** this is an in-sample backtest (the model was fitted on all matches at once), so real out-of-sample performance is likely somewhat weaker. A walk-forward backtest is a noted next step.

In [31]:
# Backtest: walk-forward evaluation of the model against real historical
# results.
#
# The honest way to do this (same principle Cup26 used): for each match in
# our completed data, predict the outcome using ONLY information that would
# have been available BEFORE that match was played, then compare to what
# actually happened. Since our attack/defense strengths were fitted using
# ALL matches at once (not walk-forward), this isn't a perfectly pure
# out-of-sample test — the model has technically "seen" every match's
# outcome during fitting. A fully rigorous walk-forward backtest would
# refit the model repeatedly as time progresses, which is expensive.
#
# For a v1, we do a simpler but still meaningful check: use the FINAL
# fitted strengths to predict every match, and score using proper scoring
# rules. This tells us how well-calibrated the model is on the data it was
# trained on — a good baseline sanity check, though a true out-of-sample
# score (like Cup26's) would likely be a bit worse than this.

def rps(probs, outcome_idx):
    # Ranked Probability Score: penalizes predictions that are further
    # from the true outcome more heavily than ones that are close.
    # outcome_idx: 0 = home win, 1 = draw, 2 = away win
    cum_probs = np.cumsum(probs)
    cum_outcome = np.cumsum([1 if i == outcome_idx else 0 for i in range(len(probs))])
    return np.mean((cum_probs - cum_outcome) ** 2)

rps_scores = []
log_losses = []
correct_picks = 0

for _, row in completed_features.iterrows():
    home, away, neutral = row["home_team"], row["away_team"], row["neutral"]
    hw, d, aw, _, _ = predict_match_dc(home, away, neutral, fitted_rho)
    probs = [hw, d, aw]

    if row["home_score"] > row["away_score"]:
        outcome_idx = 0
    elif row["home_score"] == row["away_score"]:
        outcome_idx = 1
    else:
        outcome_idx = 2

    rps_scores.append(rps(probs, outcome_idx))
    log_losses.append(-np.log(max(probs[outcome_idx], 1e-10)))

    predicted_outcome = np.argmax(probs)
    if predicted_outcome == outcome_idx:
        correct_picks += 1

print(f"Matches evaluated: {len(rps_scores)}")
print(f"Average RPS: {np.mean(rps_scores):.4f}  (lower is better; coin-flip baseline ≈ 0.24)")
print(f"Average log-loss: {np.mean(log_losses):.4f}")
print(f"Correct result picks: {correct_picks}/{len(rps_scores)} ({correct_picks/len(rps_scores)*100:.1f}%)")

Matches evaluated: 3622
Average RPS: 0.1021  (lower is better; coin-flip baseline ≈ 0.24)
Average log-loss: 0.8087
Correct result picks: 2276/3622 (62.8%)


## 31. Monte Carlo simulation of the Round of 16

Cross-checks the closed-form probabilities and handles the fact that knockout matches can't end in a draw (resolved via simulated penalty shootouts, modeled as a coin flip).

In [32]:
# Monte Carlo simulation of the current Round of 16.
#
# Why simulate instead of just using the closed-form probabilities we
# already calculated? Two reasons:
#   1. Sanity check — running many random trials should reproduce the same
#      win/draw/loss percentages we got analytically. If it doesn't match,
#      something's wrong in one of the two implementations.
#   2. Knockout matches can't end in a draw — if scores are level after
#      normal time, we need a tiebreak (extra time doesn't change the
#      scoreline distribution much in practice, so we simplify by sending
#      draws straight to penalties). We need a penalty-shootout win
#      probability, which we can estimate from your shootouts.csv file.

# First, estimate a simple penalty shootout win probability. Real shootouts
# are close to a coin flip regardless of team strength (a well-documented
# football finding — shootouts are mostly noise, not skill), so unless we
# have strong evidence otherwise, 50/50 is a reasonable, defensible default.

N_SIMULATIONS = 20000

def simulate_knockout_match(home_team, away_team, neutral, rho, rng):
    home_attack = team_strength_lookup.loc[home_team, "attack"]
    home_defense = team_strength_lookup.loc[home_team, "defense"]
    away_attack = team_strength_lookup.loc[away_team, "attack"]
    away_defense = team_strength_lookup.loc[away_team, "defense"]

    effective_home_advantage = 1.0 if neutral else home_advantage
    home_expected = avg_goals * home_attack * away_defense * effective_home_advantage
    away_expected = avg_goals * away_attack * home_defense

    home_goals = rng.poisson(home_expected)
    away_goals = rng.poisson(away_expected)

    if home_goals > away_goals:
        return home_team
    elif away_goals > home_goals:
        return away_team
    else:
        # Draw in a knockout match: goes to penalties, modeled as a coin
        # flip (shootouts show little evidence of being predictable by
        # team strength in the football analytics literature)
        return home_team if rng.random() < 0.5 else away_team

rng = np.random.default_rng(seed=42)  # fixed seed for reproducibility

print(f"{'Match':<30} {'Home Win %':>11} {'Away Win %':>11}   (includes penalty outcomes)")
for _, row in upcoming_fixtures.iterrows():
    home, away, neutral = row["home_team"], row["away_team"], row["neutral"]
    home_wins = 0
    for _ in range(N_SIMULATIONS):
        winner = simulate_knockout_match(home, away, neutral, fitted_rho, rng)
        if winner == home:
            home_wins += 1
    matchup = f"{home} vs {away}"
    print(f"{matchup:<30} {home_wins/N_SIMULATIONS*100:>10.1f}% {(1-home_wins/N_SIMULATIONS)*100:>10.1f}%")

Match                           Home Win %  Away Win %   (includes penalty outcomes)
Brazil vs Norway                     54.3%       45.7%
Mexico vs England                    35.0%       65.0%
Portugal vs Spain                    38.8%       61.2%
United States vs Belgium             34.9%       65.1%
Argentina vs Egypt                   78.3%       21.7%
Switzerland vs Colombia              40.2%       59.8%
France vs Morocco                    57.9%       42.1%


## 32. Multi-round simulation: Round of 16 through Quarterfinals

Uses the real, confirmed 2026 World Cup bracket structure (not guessed) to simulate which teams reach the semifinals.

In [33]:
# Multi-round Monte Carlo simulation: Round of 16 through Quarterfinals,
# using the REAL confirmed bracket structure (not guessed).
#
# Structure, confirmed via search:
#   QF1: France vs Morocco (already decided — no R16 needed for this one)
#   QF2: (Portugal vs Spain winner) vs (USA vs Belgium winner)
#   QF3: (Brazil vs Norway winner) vs (Mexico vs England winner)
#   QF4: (Argentina vs Egypt winner) vs (Switzerland vs Colombia winner)
#
# All quarterfinal and later matches are at neutral venues (standard for
# World Cup knockout stages beyond the round where a host is still playing
# their "home" fixture) — worth double-checking this assumption, but it's
# the standard convention.

def simulate_tournament_run(rng):
    # Round of 16 winners (France/Morocco already decided in real life)
    r16_winners = {}
    r16_winners["France_Morocco"] = simulate_knockout_match("France", "Morocco", True, fitted_rho, rng)
    r16_winners["Portugal_Spain"] = simulate_knockout_match("Portugal", "Spain", True, fitted_rho, rng)
    r16_winners["USA_Belgium"] = simulate_knockout_match("United States", "Belgium", False, fitted_rho, rng)
    r16_winners["Brazil_Norway"] = simulate_knockout_match("Brazil", "Norway", True, fitted_rho, rng)
    r16_winners["Mexico_England"] = simulate_knockout_match("Mexico", "England", False, fitted_rho, rng)
    r16_winners["Argentina_Egypt"] = simulate_knockout_match("Argentina", "Egypt", True, fitted_rho, rng)
    r16_winners["Switzerland_Colombia"] = simulate_knockout_match("Switzerland", "Colombia", True, fitted_rho, rng)

    # Quarterfinals, using the real bracket pairing
    qf1_winner = r16_winners["France_Morocco"]  # France vs Morocco is itself the QF
    qf2_winner = simulate_knockout_match(r16_winners["Portugal_Spain"], r16_winners["USA_Belgium"], True, fitted_rho, rng)
    qf3_winner = simulate_knockout_match(r16_winners["Brazil_Norway"], r16_winners["Mexico_England"], True, fitted_rho, rng)
    qf4_winner = simulate_knockout_match(r16_winners["Argentina_Egypt"], r16_winners["Switzerland_Colombia"], True, fitted_rho, rng)

    return [qf1_winner, qf2_winner, qf3_winner, qf4_winner]

# Run many full tournament simulations and count how often each team
# reaches the semifinals (i.e. wins their quarterfinal)
from collections import Counter

semifinalist_counts = Counter()
N_RUNS = 20000

for _ in range(N_RUNS):
    semifinalists = simulate_tournament_run(rng)
    for team in semifinalists:
        semifinalist_counts[team] += 1

print("Probability of reaching the semifinals (this side of the bracket):")
for team, count in sorted(semifinalist_counts.items(), key=lambda x: -x[1]):
    print(f"  {team:<15} {count/N_RUNS*100:.1f}%")

Probability of reaching the semifinals (this side of the bracket):
  France          58.3%
  Argentina       50.6%
  Spain           46.2%
  Morocco         41.7%
  England         39.0%
  Colombia        27.7%
  Brazil          26.6%
  Portugal        25.1%
  Belgium         22.8%
  Norway          20.6%
  Switzerland     15.3%
  Mexico          13.8%
  Egypt           6.5%
  United States   5.9%
